In [8]:
import os
import shutil
import pandas as pd
import re
import arcpy

EO_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\EO\EO"
PCA_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\Site\PCA"
county_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\Hyperlink\counties.shp"
counties = ["Garfield", "Mesa", "Delta", "Montrose", "Gunnison", "Lake", "Chaffee", "Grand", "Pitkin"]
counties = ["Jackson", "Larimer", "Park", "Teller", "Fremont", "El Paso", "Custer", "Pueblo", "Huerfano", "Las Animas"]
counties = [county.upper() for county in counties]

root_path = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026"

# Function to intersect layer with counties and copy files listed in the Hyperlink field
def copy_files_from_layer(layer_path, out_base):
	try:
		import arcpy
	except Exception as e:
		print('arcpy is not available in this environment:', e)
		return
	arcpy.env.overwriteOutput = True
	lyr_name = 'temp_layer'
	try:
		arcpy.MakeFeatureLayer_management(layer_path, lyr_name)
		if out_base == "EO":
			# filter to MajorGroup values of interest
			arcpy.SelectLayerByAttribute_management(lyr_name, 'NEW_SELECTION', "MajorGroup = 'Vascular Plants' OR MajorGroup = 'Natural Communities'")
		# intersect with counties shapefile (creates in-memory output)
		intersect_out = 'in_memory/eo_county_intersect'
		arcpy.Intersect_analysis([lyr_name, county_layer], intersect_out)
	except Exception as ex:
		print('Error preparing/intersecting layer:', ex)
		try:
			arcpy.Delete_management(lyr_name)
		except:
			pass
		return
	fields = ['COUNTY', 'Hyperlink']
	copied = 0
	missing = 0
	with arcpy.da.SearchCursor(intersect_out, fields) as cursor:
		for county, hyperlink in cursor:
			if not hyperlink:
				continue
			if county not in counties:
				continue
			# normalize hyperlink (remove leading slashes) and build source path
			rel = str(hyperlink).lstrip('\\/')
			src = os.path.join(root_path, rel)
			dest_dir = os.path.join(out_base, county)
			os.makedirs(dest_dir, exist_ok=True)
			try:
				shutil.copy(src, dest_dir)
				copied += 1
			except FileNotFoundError:
				print(f'Missing file: {src}')
				missing += 1
			except Exception as e:
				print(f'Failed to copy {src}: {e}')
	print(f'Layer {layer_path} -> {out_base}: copied={copied}, missing={missing}')
	try:
		arcpy.Delete_management(lyr_name)
		arcpy.Delete_management(intersect_out)
	except:
		pass

# Run for EO and PCA layers
copy_files_from_layer(EO_layer, 'EO')
copy_files_from_layer(PCA_layer, 'PCA')





Layer N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\EO\EO -> EO: copied=11229, missing=0
Layer N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\Site\PCA -> PCA: copied=973, missing=0



1. filter EO_layer to MajorGroup IN ["Vascular Plants", "Natural Communities"]
2. use arcpy to intersect EO_layer with county_layer
3. filter EO_layer to CountyName IN counties
4. for each filepath in EO_layer field called Hyperlink, copy_path = os.join(root_path, filepath), then copy file at copy_path to EO/{CountyName}/{copy_path}
5. repeat steps 2-4 for PCA_layer instead of EO_layer

See filepaths below.

In [ ]:
import os
import shutil
import pandas as pd
import re

# Paths

EO_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\EO\EO"
PCA_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\CNHPData20260424.gdb\Site\PCA"
county_layer = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026\Hyperlink\counties.shp"
counties = ["Garfield", "Mesa", "Delta", "Montrose", "Gunnison", "Lake", "Chaffee", "Grand", "Pitkin"]
root_path = r"N:\Research\CNHP\GIS_Data\Colorado_Protected_Data\CNHP\CNHP_Hyperlink2026"

